baseline code 

In [29]:
# -*- coding: utf-8 -*-
# NOTEBOOK : TEST MODÈLES SANS ÉQUILIBRAGE (SUR DATASET "features_selectionnes_v3")

# =============================================================================
# CELLULE 1 : IMPORTS ET CONFIGURATION
# =============================================================================
print("--- Cellule 1: Imports et Configuration ---")

import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from collections import Counter # Ajouté pour la distribution de la cible

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, auc, average_precision_score
)

# --- Configuration des Chemins et Variables ---
INPUT_FILE = "/home/kaoutar/Downloads/dataset_features_selectionnes_avec_appetence3.csv"
TARGET_COL_NAME = 'TARGET' # Confirmé par votre liste de colonnes
RANDOM_STATE = 42
# ---------------------------------------------

print("-" * 50)
print("Imports et Configuration de Base Terminés.")
print(f"Utilisation du fichier d'entrée : {INPUT_FILE}")
print(f"Colonne Cible Attendue : {TARGET_COL_NAME}")
print(f"La colonne 'SCORE_APPETENCE' sera ignorée pour la création de X.")
print("-" * 50)
# --- FIN Cellule 1 ---

--- Cellule 1: Imports et Configuration ---
--------------------------------------------------
Imports et Configuration de Base Terminés.
Utilisation du fichier d'entrée : /home/kaoutar/Downloads/dataset_features_selectionnes_avec_appetence3.csv
Colonne Cible Attendue : TARGET
La colonne 'SCORE_APPETENCE' sera ignorée pour la création de X.
--------------------------------------------------



--- Chargement du jeu de données: /home/kaoutar/Downloads/dataset_features_selectionnes_avec_appetence3.csv ---
Tentative de chargement de /home/kaoutar/Downloads/dataset_features_selectionnes_avec_appetence3.csv avec sep=',' et engine='python'...
Données chargées. Shape initiale: (1059691, 34)
Extrait des colonnes détectées: ['RADICAL', 'MONTANT_TOTAL_sum_x', 'MONTANT_TOTAL_mean_x', 'MONTANT_TOTAL_count_x', 'NOMBRE_OPE_sum', 'NOMBRE_OPE_mean', 'DT_CONFIRMATION_max', 'DT_CONFIRMATION_min', 'TRANSACTION_RECENCY_DAYS', 'TRANSACTION_FREQUENCY_DAYS']...
Colonnes initiales dans X (avant nettoyage X): ['RADICAL', 'MONTANT_TOTAL_sum_x', 'MONTANT_TOTAL_mean_x', 'MONTANT_TOTAL_count_x', 'NOMBRE_OPE_sum', 'NOMBRE_OPE_mean', 'DT_CONFIRMATION_max', 'DT_CONFIRMATION_min', 'TRANSACTION_RECENCY_DAYS', 'TRANSACTION_FREQUENCY_DAYS']...
Shape X avant filtrage variance: (1059691, 33)

Préparation X/y OK. Shape X final: (1059691, 33), Shape y: (1059691,)
Distribution de la cible y: 
TARGET
0    0.924043


In [30]:
# =============================================================================
# CELLULE 2 : CHARGEMENT DES DONNÉES ET PRÉPARATION X, y
# =============================================================================
print(f"\n--- Chargement du jeu de données: {INPUT_FILE} ---")

X = None
y = None
data_loaded_successfully = False

if os.path.exists(INPUT_FILE):
    try:
        print(f"Tentative de chargement de {INPUT_FILE} avec sep=',' et engine='python'...")
        df_source = pd.read_csv(INPUT_FILE, sep=',', engine='python')
        print(f"Données chargées. Shape initiale: {df_source.shape}")
        print(f"Extrait des colonnes détectées: {df_source.columns.tolist()[:10]}...")

        if TARGET_COL_NAME not in df_source.columns:
             raise ValueError(f"Colonne cible '{TARGET_COL_NAME}' manquante.")
        if 'SCORE_APPETENCE' not in df_source.columns:
            print("INFO: La colonne 'SCORE_APPETENCE' n'est pas dans le fichier, ce qui est attendu si elle ne doit pas être utilisée.")
            cols_to_drop_for_X = [TARGET_COL_NAME]
        else:
            print("INFO: La colonne 'SCORE_APPETENCE' est présente et sera retirée des features X.")
            cols_to_drop_for_X = [TARGET_COL_NAME, 'SCORE_APPETENCE']


        # --- MODIFICATION ICI pour retirer SCORE_APPETENCE de X ---
        X = df_source.drop(columns=cols_to_drop_for_X, errors='ignore') # errors='ignore' si SCORE_APPETENCE n'existe pas
        y = df_source[TARGET_COL_NAME]
        # ---------------------------------------------------------

        print(f"Colonnes initiales dans X (après drop cibles): {X.columns.tolist()[:10]}...")
        X = X.select_dtypes(include=[np.number]) # Garder seulement numériques
        print(f"Colonnes numériques conservées dans X: {X.columns.tolist()[:10]}...")


        if X.isna().sum().sum() > 0:
            print(f"ATTENTION: {X.isna().sum().sum()} NaNs détectés dans X. Remplissage par 0...")
            X = X.fillna(0)
        if y.isna().any():
            print(f"ATTENTION: NaNs détectés dans y. Suppression des lignes correspondantes...")
            valid_indices = ~y.isna()
            X = X.loc[valid_indices]
            y = y.loc[valid_indices]
            print(f"  Shape après suppression NaNs cible - X: {X.shape}, y: {y.shape}")

        if not pd.api.types.is_numeric_dtype(y):
            try: y = y.astype(int)
            except ValueError: raise ValueError(f"Cible '{TARGET_COL_NAME}' non convertible en int.")
        unique_y = np.unique(y)
        if not set(unique_y).issubset({0, 1}):
            print(f"Warning: Cible non binaire: {unique_y}. Binarisation (val > 0 -> 1)...")
            y = (y > 0).astype(int)
            print(f"Distribution cible après binarisation: {Counter(y)}")

        print(f"Shape X avant filtrage variance: {X.shape}")
        if not X.empty:
            cols_before_var_filt = X.columns
            variance = X.var()
            cols_to_keep = variance[(variance > 1e-6) & variance.notna()].index
            X = X[cols_to_keep].copy()
            # ... (reste du filtre de variance)
            cols_after_var_filt = X.columns
            dropped_cols_var = cols_before_var_filt.difference(cols_after_var_filt)
            if not dropped_cols_var.empty:
                print(f"  Colonnes à variance nulle/quasi-nulle supprimées: {dropped_cols_var.tolist()}")
        else: print("X est vide avant le filtre de variance.")

        print(f"\nPréparation X/y OK. Shape X final: {X.shape}, Shape y: {y.shape}")
        print(f"Distribution de la cible y: \n{y.value_counts(normalize=True)}")
        data_loaded_successfully = True

    except Exception as e:
        print(f"ERREUR lors du chargement ou préparation initiale: {e}")
        data_loaded_successfully = False
else:
    print(f"ERREUR: Fichier {INPUT_FILE} non trouvé.")
    data_loaded_successfully = False

# --- FIN Cellule 2 ---


--- Chargement du jeu de données: /home/kaoutar/Downloads/dataset_features_selectionnes_avec_appetence3.csv ---
Tentative de chargement de /home/kaoutar/Downloads/dataset_features_selectionnes_avec_appetence3.csv avec sep=',' et engine='python'...
Données chargées. Shape initiale: (1059691, 34)
Extrait des colonnes détectées: ['RADICAL', 'MONTANT_TOTAL_sum_x', 'MONTANT_TOTAL_mean_x', 'MONTANT_TOTAL_count_x', 'NOMBRE_OPE_sum', 'NOMBRE_OPE_mean', 'DT_CONFIRMATION_max', 'DT_CONFIRMATION_min', 'TRANSACTION_RECENCY_DAYS', 'TRANSACTION_FREQUENCY_DAYS']...
INFO: La colonne 'SCORE_APPETENCE' est présente et sera retirée des features X.
Colonnes initiales dans X (après drop cibles): ['RADICAL', 'MONTANT_TOTAL_sum_x', 'MONTANT_TOTAL_mean_x', 'MONTANT_TOTAL_count_x', 'NOMBRE_OPE_sum', 'NOMBRE_OPE_mean', 'DT_CONFIRMATION_max', 'DT_CONFIRMATION_min', 'TRANSACTION_RECENCY_DAYS', 'TRANSACTION_FREQUENCY_DAYS']...
Colonnes numériques conservées dans X: ['RADICAL', 'MONTANT_TOTAL_sum_x', 'MONTANT_TOTAL

In [31]:
# =============================================================================
# CELLULE 3 : ÉTAPE V - PRÉPARATION AVANT MODÉLISATION (SANS ÉQUILIBRAGE EXPLICITE)
# =============================================================================
# ... (Ce code reste identique à la version précédente que je vous ai donnée pour cette cellule)
# Il utilisera X et y préparés par la Cellule 2.

X_train_scaled, X_test_scaled, y_train, y_test = None, None, None, None # Initialiser
data_ready_for_modeling = False

if data_loaded_successfully and X is not None and y is not None and not X.empty:
    print("\n--- ÉTAPE V: Préparation Avant Modélisation ---")
    print("Séparation Train/Test (80/20)...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                        random_state=RANDOM_STATE, stratify=y)
    print(f"  Shapes - Train: {X_train.shape}, Test: {X_test.shape}")
    print("\nPas de rééchantillonnage appliqué aux données d'entraînement (utilisation de y_train original).")
    y_train_final = y_train.copy()
    X_train_processed = X_train.copy()

    print("\nMise à l'échelle (StandardScaler)...")
    numerical_cols = X.select_dtypes(include=np.number).columns
    binary_cols = [col for col in X.select_dtypes(include='int').columns if X[col].nunique(dropna=False) <= 2]
    age_group_cols = [col for col in X.columns if col.startswith('age_group_')]
    cols_to_exclude_from_scaling = binary_cols + age_group_cols
    numerical_cols_to_scale = [c for c in numerical_cols if c not in cols_to_exclude_from_scaling and c in X_train_processed.columns]

    scaler = StandardScaler()
    X_train_scaled = X_train_processed.copy()
    X_test_scaled = X_test.copy()

    if numerical_cols_to_scale:
         if X_train_scaled[numerical_cols_to_scale].isna().sum().sum() > 0:
             X_train_scaled.loc[:, numerical_cols_to_scale] = X_train_scaled[numerical_cols_to_scale].fillna(X_train[numerical_cols_to_scale].median())
         if X_test_scaled[numerical_cols_to_scale].isna().sum().sum() > 0:
             X_test_scaled.loc[:, numerical_cols_to_scale] = X_test_scaled[numerical_cols_to_scale].fillna(X_train[numerical_cols_to_scale].median())
         print(f"  Scaling des colonnes ({len(numerical_cols_to_scale)}): {numerical_cols_to_scale[:5]}...")
         X_train_scaled.loc[:, numerical_cols_to_scale] = scaler.fit_transform(X_train_scaled[numerical_cols_to_scale])
         X_test_scaled.loc[:, numerical_cols_to_scale] = scaler.transform(X_test_scaled[numerical_cols_to_scale])
         print("  Scaling appliqué.")
    else:
         print("  Aucune colonne appropriée à scaler n'a été trouvée.")
    data_ready_for_modeling = True
else:
    print("Erreur: Données X ou y non chargées ou X vide. Arrêt avant Étape V.")
# --- FIN Cellule 3 ---


--- ÉTAPE V: Préparation Avant Modélisation ---
Séparation Train/Test (80/20)...
  Shapes - Train: (847752, 32), Test: (211939, 32)

Pas de rééchantillonnage appliqué aux données d'entraînement (utilisation de y_train original).

Mise à l'échelle (StandardScaler)...
  Scaling des colonnes (32): ['RADICAL', 'MONTANT_TOTAL_sum_x', 'MONTANT_TOTAL_mean_x', 'MONTANT_TOTAL_count_x', 'NOMBRE_OPE_sum']...
  Scaling appliqué.


/tmp/ipykernel_5887/1377971509.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.57104604  0.84178778  1.00852962 ... -0.89434748  0.51816612
  0.29376946]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_train_scaled.loc[:, numerical_cols_to_scale] = scaler.fit_transform(X_train_scaled[numerical_cols_to_scale])
/tmp/ipykernel_5887/1377971509.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.29735591  1.15354954 -0.33400164 ...  0.90754867 -0.88281128
 -0.10007978]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_test_scaled.loc[:, numerical_cols_to_scale] = scaler.transform(X_test_scaled[numerical_cols_to_scale])


In [32]:
# =============================================================================
# CELLULE 4 : ÉTAPE VI - ENTRAÎNEMENT ET ÉVALUATION (SANS ÉQUILIBRAGE EXPLICITE)
# =============================================================================
# ... (Ce code reste identique à la version précédente que je vous ai donnée pour cette cellule)
# Il entraînera LogisticRegression, RandomForest, GradientBoosting sans class_weight
# et affichera leurs performances.

if data_ready_for_modeling:
    print(f"\n--- ÉTAPE VI: Entraînement et Évaluation (Sur Données Originales Déséquilibrées) ---")
    model_configs_no_balance = {
        "LogisticRegression": LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, solver='liblinear'),
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, max_depth=15, min_samples_leaf=5),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE, max_depth=5)
    }
    results_current_run = {}
    for name, model in model_configs_no_balance.items():
        if model is None: continue
        print(f"\n--- Modèle : {name} (Sans Équilibrage Explicite) ---")
        try:
            start_time = datetime.now()
            print(f"  Entraînement {name}...")
            model.fit(X_train_scaled.fillna(0), y_train_final)
            train_time = datetime.now() - start_time
            print(f"  Entraînement terminé en {train_time}")

            print(f"  Évaluation {name} sur Test Set...")
            X_test_eval = X_test_scaled.fillna(0)
            y_pred_test = model.predict(X_test_eval)
            y_proba_test = model.predict_proba(X_test_eval)[:, 1]

            print(f"\n  Classification Report (Test - {name}):")
            print(classification_report(y_test, y_pred_test, zero_division=0))
            print(f"\n  Matrice de Confusion (Test - {name}):")
            cm_model = confusion_matrix(y_test, y_pred_test)
            print(cm_model)
            roc_auc = roc_auc_score(y_test, y_proba_test) if len(np.unique(y_test)) > 1 else 0.5
            precision, recall, _ = precision_recall_curve(y_test, y_proba_test)
            pr_auc = auc(recall, precision)
            report_dict = classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)
            print(f"\n  AUC ROC: {roc_auc:.4f}, AUC PR: {pr_auc:.4f}")
            results_current_run[name] = {
                 'AUC_ROC': roc_auc, 'AUC_PR': pr_auc,
                 'F1_Class1': report_dict.get('1', {}).get('f1-score', 0),
                 'Recall_Class1': report_dict.get('1', {}).get('recall', 0),
                 'Precision_Class1': report_dict.get('1', {}).get('precision', 0)
            }
            # try: # Affichage heatmap
            #     plt.figure(figsize=(6,4)); sns.heatmap(cm_model, annot=True, fmt='d', cmap='Oranges'); plt.title(f'Matrice de Confusion - {name} (Sans Équilibrage)'); plt.show()
            # except Exception as e: print(f"  Erreur affichage heatmap: {e}")
        except Exception as e:
            print(f"  ERREUR pour {name}: {e}")
            results_current_run[name] = {'AUC_ROC': np.nan, 'AUC_PR': np.nan, 'F1_Class1': np.nan}
    if len(results_current_run) > 0:
        print("\n\n--- Résultats Finaux de cette Exécution (Sans Équilibrage Explicite) ---")
        results_df_current_run = pd.DataFrame(results_current_run).T
        print(results_df_current_run)
else:
    print("\nErreur: Données non prêtes pour l'étape VI.")
# --- FIN Cellule 4 ---


--- ÉTAPE VI: Entraînement et Évaluation (Sur Données Originales Déséquilibrées) ---

--- Modèle : LogisticRegression (Sans Équilibrage Explicite) ---
  Entraînement LogisticRegression...
  Entraînement terminé en 0:00:20.662902
  Évaluation LogisticRegression sur Test Set...

  Classification Report (Test - LogisticRegression):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    195841
           1       1.00      0.97      0.98     16098

    accuracy                           1.00    211939
   macro avg       1.00      0.98      0.99    211939
weighted avg       1.00      1.00      1.00    211939


  Matrice de Confusion (Test - LogisticRegression):
[[195841      0]
 [   548  15550]]

  AUC ROC: 0.9989, AUC PR: 0.9912

--- Modèle : RandomForest (Sans Équilibrage Explicite) ---
  Entraînement RandomForest...
  Entraînement terminé en 0:00:46.168500
  Évaluation RandomForest sur Test Set...

  Classification Report (Test - RandomFores

In [28]:
df_check_new = pd.read_csv("/home/kaoutar/Downloads/dataset_features_selectionnes_avec_appetence3.csv", sep=',') # ou '\t' si besoin
print(df_check_new.columns.tolist())
print(df_check_new.shape)

['RADICAL', 'MONTANT_TOTAL_sum_x', 'MONTANT_TOTAL_mean_x', 'MONTANT_TOTAL_count_x', 'NOMBRE_OPE_sum', 'NOMBRE_OPE_mean', 'DT_CONFIRMATION_max', 'DT_CONFIRMATION_min', 'TRANSACTION_RECENCY_DAYS', 'TRANSACTION_FREQUENCY_DAYS', 'IS_CONNECTED', 'CONNECTIONCOUNT', 'CONNECTION_RECENCY_DAYS', 'ACCOUNT_AGE_DAYS', 'IS_VALID', 'MONTANT_std', 'NOMBRE_OPERATION_sum', 'BRANCH_AVG_AMOUNT_PER_OP', 'MONTANT_TOTAL_mean_y', 'CHANNEL_MOBILE', 'CHANNEL_WEB', 'TYPE_FACTURE', 'TYPE_VIR', 'MONTANT_TOTAL_mean_x_X_DT_CONFIRMATION_max', 'MONTANT_TOTAL_mean_x_PLUS_DT_CONFIRMATION_max', 'MONTANT_TOTAL_mean_x_MINUS_DT_CONFIRMATION_max', 'MONTANT_TOTAL_mean_x_X_DT_CONFIRMATION_min', 'MONTANT_TOTAL_mean_x_PLUS_DT_CONFIRMATION_min', 'MONTANT_TOTAL_mean_x_MINUS_DT_CONFIRMATION_min', 'DT_CONFIRMATION_max_X_DT_CONFIRMATION_min', 'DT_CONFIRMATION_max_PLUS_DT_CONFIRMATION_min', 'DT_CONFIRMATION_max_MINUS_DT_CONFIRMATION_min', 'SCORE_APPETENCE', 'TARGET']
(1059691, 34)
